In [ ]:
# from simple_battery_pyomo import solve_pyomo
from pv_battery_pyomo import solve_pyomo_pv
from pyomo_generalization import solve_pyomo_generalized
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from utils import *
pd.options.plotting.backend = "plotly"
import time

In [ ]:
# Measured consumption of a household (15 min, kW). load_besnica.csv already includes a PV plant, use it with pv_size = 0
df_model  = pd.read_csv('../data/load_sencur.csv')
df_model['date_time'] = pd.to_datetime(df_model['date_time'])
df_model = df_model.set_index('date_time')
df_model.head()

In [ ]:
pv_data_original = pd.read_csv('../data/pv_data.csv', sep=';', decimal=',')
pv_data_original.date_time = pd.to_datetime(pv_data_original.date_time)
pv_data_original.set_index("date_time", inplace=True)
pv_data_original = pv_data_original[~pv_data_original.index.duplicated(
            keep='first')]

# Day-ahead prices (EUR/MWh), date_time is the end of the interval. Update with download_prices.py
prices_original = pd.read_csv('../data/day_ahead_prices.csv', parse_dates=['date_time'])
prices_original.set_index('date_time', inplace=True)

# Determine the common start and end between df_model, pv_data_original and prices_original
end = min(df_model.index[-1], pv_data_original.index[-1], prices_original.index[-1])
start = max(end - pd.DateOffset(years=1), df_model.index[0],
            pv_data_original.index[0])
# Filter both dataframes to this range

df_model = df_model.loc[start:end]
display(pv_data_original.head())
# Resample both dataframes to 15-minute intervals, using forward-fill to handle missing data if necessary
pv_data_original = pv_data_original.resample(
    '15min').ffill()  # '15min' represents 15 minutes
pv_data_original.head()

In [ ]:
pv_size = 15# kW
pv_operating_hours = 1100
pv_data = pv_data_original.loc[start:end] * pv_size * pv_operating_hours / 1000
df_model = df_model[~df_model.index.duplicated(keep='first')]
df_model = df_model.resample('15min').mean()
# Missing days are filled with the same time one week before
df_model = df_model.fillna(df_model.shift(96 * 7)).ffill()
# Align the indices of both dataframes, ensuring they are consistent
common_index = df_model.index.intersection(pv_data.index)
df_model = df_model.loc[common_index]
pv_data = pv_data.loc[common_index]
production = pv_data['p'].values

In [ ]:
df_prices = prices_original[['price']]
df_prices = df_prices[~df_prices.index.duplicated(
            keep='first')]
# Hourly prices are labelled with the end of the hour, so the 15 min intervals take the next value
df_prices = df_prices.resample(
    '15min').bfill()

prices_data = df_prices.loc[common_index]
display(prices_data.head())
da_prices_original = prices_data['price'].values

In [ ]:
plt.plot(da_prices_original)

In [ ]:
BATTERY_SIZE = 20  # kWh
BATTERY_POWER = 20  # kW

In [ ]:
dates_original = np.array(list(pd.to_datetime(df_model.index)))
blocks_5d = individual_tariff_times(dates_original)
times_energy_blocks_original = np.zeros_like(dates_original)
capacity = BATTERY_SIZE  # kWh
power = BATTERY_POWER  # kW
battery_price = capacity*100
battery_life_cycles = 2000
for i in range(5):
    times_energy_blocks_original += blocks_5d[i] * (i + 1)
load_original = list(df_model["p"])
# energy_buy_prices_original = np.ones_like(dates_original)*0.13
# energy_sell_prices_original = np.ones_like(dates_original)*0.07
energy_buy_prices_original = list(da_prices_original /1000)
energy_sell_prices_original = list(da_prices_original /1000)
production_original = list(pv_data["p"])
block_power_prices = {1: 5.33444, 2: 1.08944, 3: 0.142570, 4: 0.00368, 5: 0.00000}
energy_grid_prices = {1: 0.01454, 2: 0.01389, 3: 0.01369, 4: 0.0133, 5: 0.01329}
power_price = 0.
# N = 1*96
# N = 96*7
# start = 50
# dates = dates_original[start:start +N]
# load = load[start:start +N]
# dates_in = range(len(dates_original))
# energy_prices = energy_prices[start:start +N]
# times_energy_blocks =times_energy_blocks[start:start +N]

In [ ]:
df_res = pd.DataFrame()
block_powers_list = []
block_energies_list = []
for i in range(len(dates_original) // (96 * 7)):  # all full weeks
    t_start = time.time()
    start = (i) * 96*7
    N = 96*7
    dates = dates_original[start:start + N]
    load = load_original[start:start + N]
    dates_in = range(len(dates))
    energy_buy_prices = energy_buy_prices_original[start:start + N]
    energy_sell_prices = energy_sell_prices_original[start:start + N]
    times_energy_blocks = times_energy_blocks_original[start:start + N]
    times_power_blocks = np.ones_like(dates) * 1
    production = production_original[start:start + N]
    # Solve the optimization problem
    return_dict, block_peaks, block_energies = solve_pyomo_generalized(
        dates_in,
        times_energy_blocks,
        times_power_blocks,
        load,
        production,
        energy_buy_prices,
        energy_sell_prices,
        BATTERY_POWER,
        BATTERY_SIZE,
        energy_blocks=list(set(times_energy_blocks)),
        power_blocks=list(set(times_power_blocks)),
        block_power_prices=block_power_prices,
        energy_grid_prices=energy_grid_prices,
        max_grid_power=70,
        min_grid_power=-10,
        battery_price=battery_price,
        battery_life_cycles=battery_life_cycles,
        soc_init=capacity * 0.9,
        min_settlement_power=5.)
    print("dates_in", dates_in)
    print("times_energy_blocks", times_energy_blocks)
    print("load", load)
    print("production", production)
    print("energy_buy_prices", energy_buy_prices)
    print("energy_sell_prices", energy_sell_prices)
    print("block_power_prices", block_power_prices)
    print("energy_grid_prices", energy_grid_prices)
    print("battery_price", battery_price)
    print("battery_life_cycles", battery_life_cycles)
    print("soc_init", capacity * 0.9)
    print("return_dict", return_dict)
    # to csv
    df = pd.DataFrame(return_dict)
    # to csv
    # df.to_csv("original_algday1.csv")
    # break
    df_res = pd.concat([df_res, pd.DataFrame(return_dict)], axis=0)
    block_powers_list.append(block_peaks)
    block_energies_list.append(block_energies)
    t_end = time.time()
    print(
        f"Iteration {dates_original[start]} took {t_end - t_start:.2f} seconds"
    )
df_res = df_res.reset_index(drop=True)

In [ ]:
df_res.head()

In [ ]:
# df_res = df
df_res["date_time"] = dates_original[:len(df_res)]
load_arr = np.array(df_res["load"])
battery_arr = np.array(df_res["power"])
soc_arr = np.array(df_res["soc"])
grid_arr = np.array(df_res["grid_power"])
production_arr = np.array(df_res["original_production"])
limited_production_arr = np.array(df_res["limited_production"])
energy_prices = np.array(df_res["energy_price"])*1000


In [ ]:
original_output = (load_arr* (load_arr < 0)).sum()/4
after_output = (grid_arr* (grid_arr < 0)).sum()/4
battery_cycles = (battery_arr* (battery_arr > 0)).sum()/4/capacity
print("Original output: ", original_output)
print("After output: ", after_output)
print("Battery cycles: ", battery_cycles)

In [ ]:
df_res["original_meter_consumption"] = df_res["load"] - df_res["original_production"]

In [ ]:
import plotly

import plotly.graph_objects as go

# Create a Plotly figure
fig = go.Figure()

# Add traces for each plot
fig.add_trace(go.Scatter(x=df_res['date_time'], y=df_res['load'], mode='lines', name='Load'))
fig.add_trace(go.Scatter(x=df_res['date_time'], y=df_res['soc'], mode='lines', name='SOC'))
fig.add_trace(go.Scatter(x=df_res['date_time'], y=df_res['power'], mode='lines', name='Battery Power'))
fig.add_trace(go.Scatter(x=df_res['date_time'], y=df_res['grid_power'], mode='lines', name='Grid Power'))
fig.add_trace(go.Scatter(x=df_res['date_time'], y=df_res['original_production'], mode='lines', name='Original Production'))
fig.add_trace(
    go.Scatter(x=df_res['date_time'],
               y=df_res['limited_production'],
               mode='lines',
               name='Limited Production'))
fig.add_trace(go.Scatter(x=df_res['date_time'], y=df_res['original_meter_consumption'], mode='lines', name='Original Consumption with Solar'))
fig.add_trace(go.Scatter(x=df_res['date_time'], y=df_res['energy_price']*100, mode='lines', name='Energy Price'))
fig.show()
#save fig
# fig.write_html("battery_simulation.html")

In [ ]:
import plotly

import plotly.graph_objects as go

# Create a Plotly figure
fig = go.Figure()

# Add traces for each plot
fig.add_trace(go.Scatter(x=dates, y=load[:N], mode='lines', name='Load'))
fig.add_trace(go.Scatter(x=dates, y=return_dict["grid_power"], mode='lines', name='Grid Power'))
fig.add_trace(go.Scatter(x=dates, y=return_dict["power"], mode='lines', name='Battery Power'))
fig.add_trace(go.Scatter(x=dates, y=return_dict["soc"], mode='lines', name='SOC'))

# Update layout for better visualization
fig.update_layout(
    title="Energy Optimization Results",
    xaxis_title="Date",
    yaxis_title="Power/SOC",
    legend_title="Legend",
    template="plotly_white"
)

# Show the plot
fig.show()
